In [57]:
import numpy as np
import os

target_folder = '/data6/Users/yeonjoon/VcbMVAStudy/TabNet_template/TabNET_model/largePhaseSpace_B_SPANet_MultiClass'
arr = np.load(f'{target_folder}/data.npz', allow_pickle=True)

In [58]:
info = np.load(f'{target_folder}/info.npy', allow_pickle=True)

In [59]:
varlist = info[()]['data_info']['varlist']
varlist.remove('weight')  # remove weight from the variable list

In [60]:
train_features = arr['train_features']
train_y = arr['train_y']
train_weight = arr['train_weight']


In [61]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import mplhep as hep

# =========================
# 사용자 설정
# =========================
hep.style.use("CMS")           # 스타일: "CMS" / "ATLAS" / "LHCb" 등
YEAR_LABEL = "Run2"            # CMS 라벨 year 표기 (예: "2016–2018", "2018", "Run2")
LUMI_STR   = None              # 예: '137 fb$^{-1}$ (13 TeV)' 또는 None
OUT_PDF    = "label_distributions_by_feature.pdf"
OUT_PDF = os.path.join(target_folder, OUT_PDF)

NUM_BINS   = 50
LOW_Q      = 0.0               # x-축 하/상위 퍼센타일 컷
HIGH_Q     = 100.0

# 필수 입력:
# - train_features: shape (N, D)
# - train_y:        shape (N,)
# - (선택) train_weight: shape (N,)
try:
    train_weight
except NameError:
    train_weight = None  # 없으면 None

# varlist가 없다면 자동 대체(피처 이름)
try:
    varlist
    _HAS_VARLIST = True
except NameError:
    _HAS_VARLIST = False

classes = np.unique(train_y)
N, D = train_features.shape

with PdfPages(OUT_PDF) as pdf:
    for j in range(D):
        x_all = train_features[:, j]

        # 유효값 마스크
        m = np.isfinite(x_all) & np.isfinite(train_y)
        if train_weight is not None:
            m &= np.isfinite(train_weight)

        x = x_all[m]
        y = train_y[m]
        w = (train_weight[m] if train_weight is not None else None)

        # 데이터 없으면 건너뜀
        if x.size == 0:
            continue

        # 퍼센타일 기반 범위
        lo, hi = np.percentile(x, [LOW_Q, HIGH_Q])
        if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
            lo, hi = np.nanmin(x), np.nanmax(x)
            if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
                lo, hi = float(np.min(x)) - 0.5, float(np.max(x)) + 0.5

        # 피처 이름
        if _HAS_VARLIST and isinstance(varlist, (list, tuple)) and len(varlist) == D:
            feat_name = str(varlist[j])
        else:
            feat_name = f"feature[{j}]"

        # Figure/Axes 생성
        fig, ax = plt.subplots(figsize=(12, 10))

        # 클래스별 히스토그램 (HEP 스타일엔 step이 깔끔)
        for c in classes:
            xm = x[y == c]
            if xm.size == 0:
                continue
            wm = (w[y == c] if w is not None else None)
            ax.hist(
                xm,
                bins=NUM_BINS,
                range=(lo, hi),
                histtype="step",
                density=True,
                weights=wm,
                label=f"class {c} (n={xm.size})",
            )

        # CMS 라벨(축 안쪽 좌상단)
        try:
            hep.cms.label(ax=ax, data=False, year=YEAR_LABEL, lumi=LUMI_STR)
        except Exception:
            # mplhep 버전 호환을 위한 안전장치
            hep.cms.text("Preliminary", ax=ax)

        # ---- 제목: 축 바깥 최상단 중앙 (겹침 X) ----
        fig.text(
            0.5, 0.99,
            f"{feat_name} — label-wise distribution",
            ha="center", va="top"
        )

        # 축/범례/그리드
        ax.set_xlabel(feat_name)
        ax.set_ylabel("Density")
        ax.grid(alpha=0.3)
        ax.legend(frameon=False, loc="best")

        # 제목 공간 확보
        plt.tight_layout(rect=(0, 0, 1, 0.96))
        pdf.savefig(fig)
        plt.close(fig)

print(f"Saved: {OUT_PDF}")

Saved: /data6/Users/yeonjoon/VcbMVAStudy/TabNet_template/TabNET_model/largePhaseSpace_B_SPANet_MultiClass/label_distributions_by_feature.pdf


In [56]:
train_features[:, 15]

array([35.20113 , 25.424532, 90.856476, ..., 53.264164, 81.77799 ,
       99.02128 ], dtype=float32)